In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Set MODEL in .env. Names starting with "gpt" use OpenAI (needs OPENAI_API_KEY);
# anything else is treated as a local Ollama model, e.g. "llama3" or "deepseek-r1:1.5b".
MODEL = os.getenv("MODEL", "llama3")

# The source PDF is downloaded here at runtime (see the Selenium cell below).
PDF_PATH = Path("data") / "TelecareMonitoringRepresentatives.pdf"
PDF_PATH.parent.mkdir(exist_ok=True)

Below we are chunking the documents in three ways to verify which one works fine
1. Recursive character Text Splitting
2. Semantic chunking (not implemented yet)
3. LLM-based agentic chunking

We will then see how each of the chunking performs and then implement the one that produces the relevant answer when evaluating

* Web-based loader for scraping is not working in this case at the moment

In [ ]:

# When performing the chunking we also perform web scraping if necessary
# from langchain_community.document_loaders import WebBaseLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(chunk_size =1000, chunk_overlap = 20)

# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
# }

# loader = WebBaseLoader("https://titusdinkar.wixsite.com/titus/projects", headers=headers)
# documents = loader.load_and_split(text_splitter)
# documents


* We will instead be using requests and beautiful soup to scrape the website and the pdf content in it. This is not detecting pdf masked buttons

In [ ]:
import requests
from bs4 import BeautifulSoup
import fitz
from langchain.text_splitter import RecursiveCharacterTextSplitter

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://titusdinkar.wixsite.com/titus/projects"
response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    # Step 2: Find PDF links (buttons often contain <a> tags with href)
    pdf_links = [a["href"] for a in soup.find_all("a", href=True) if a["href"].startswith("5. Telecare")]

    if not pdf_links:
        print("No PDF links found on the page.")
    else:
        for pdf_url in pdf_links:
            print(f"Found PDF: {pdf_url}")

            # Step 3: Download the PDF
            pdf_response = requests.get(pdf_url, headers=headers)
            pdf_filename = "Telecare Monitoring Process document (pdf)"

            with open(pdf_filename, "wb") as f:
                f.write(pdf_response.content)

            print(f"PDF downloaded: {pdf_filename}")

            # Step 4: Extract text from the PDF
            pdf_text = ""
            with fitz.open(pdf_filename) as pdf_doc:
                for page in pdf_doc:
                    pdf_text += page.get_text("text") + "\n"

            # Step 5: Split text into chunks
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
            documents = text_splitter.split_text(pdf_text)

            print(documents)  # This will print the extracted PDF content
else:
    print(f"Failed to load page. Status code: {response.status_code}")


In the code below I used selenium webdriver to detect links within the website

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

# Setup Selenium WebDriver (make sure you have Chrome installed)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get(url)

# Give the page some time to load dynamic content
time.sleep(2)

# Find all buttons that might link to PDFs and click them
# Assuming the buttons have "PDF" in their text or are specific to PDF links
pdf_buttons = driver.find_elements(By.XPATH, "//a[contains(text(), 'Telecare Monitoring Process document (.pdf)') or contains(@href, '.pdf')]")

# Extract the PDF URLs from these buttons
pdf_urls = [button.get_attribute("href") for button in pdf_buttons if ".pdf" in button.get_attribute("href")]

# If PDF URLs are found, download and extract content
pdf_text = ""
if pdf_urls:
    print("Found PDF Links:", pdf_urls)
    # Step 3: Download and extract text from the PDF
    pdf_url = pdf_urls[0]  # there's only one PDF at the moment
    pdf_response = requests.get(pdf_url, headers=headers)
    pdf_filename = PDF_PATH
    
    with open(pdf_filename, "wb") as f:
        f.write(pdf_response.content)

    print(f"PDF downloaded: {pdf_filename}")

    # Extract text from the downloaded PDF
    with fitz.open(pdf_filename) as pdf_doc:
        for page in pdf_doc:
            pdf_text += page.get_text("text") + "\n"

    print(f"Extracted text from PDF: {pdf_text[:500]}...")  # Preview first 500 characters
else:
    print("No PDF links found.")

# Close the browser once done
driver.quit()


* Using the objects defined in the earlier steps

In [ ]:
# Extract the text (excluding unwanted tags like scripts)
website_text = ' '.join([element.get_text() for element in soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])])

* The code below creates chunks of text within the website

In [ ]:
# 4. Combine website text and PDF text and chunk them for further processing
combined_text = website_text + "\n" + pdf_text

# Step 5: Chunk the combined text using langchain's RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
documents = text_splitter.split_text(combined_text)

print(f"Number of chunks: {len(documents)}")
print("chunk:", documents)

* The code below creates chunks of all the buttons that contain pdf links

In [ ]:
# Step 1: Chunk website content separately
website_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
website_chunks = website_text_splitter.split_text(website_text)

# Step 2: Chunk PDF content separately
pdf_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
pdf_chunks = pdf_text_splitter.split_text(pdf_text)

# Combine both sets of chunks (website and PDF) into one list
combined_chunks = website_chunks + pdf_chunks

* Print results of chunks created from website chunks and pdf chunks

In [ ]:
# Output results for all chunks
print(f"Total number of website chunks: {len(website_chunks)}")
print(f"Total number of PDF chunks: {len(pdf_chunks)}")
print(f"Total number of combined chunks: {len(combined_chunks)}\n")

# Print all website chunks
print("Website Chunks:")
for i, chunk in enumerate(website_chunks, 1):
    print(f"Website Chunk {i}:")
    print(chunk)
    print("-" * 50)  # Divider between chunks

In [ ]:
# Print all PDF chunks
print("PDF Chunks:")
for i, chunk in enumerate(pdf_chunks):
    print(f"PDF Chunk {i}:")
    print(chunk)
    print("-" * 50)  # Divider between chunks

In [ ]:
pdf_chunks[4]

In [ ]:
website_chunks[24]

In [ ]:
import openai
from langchain_openai.chat_models import ChatOpenAI
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM
from langchain_ollama import OllamaEmbeddings
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import MultiQueryRetriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings



if MODEL.startswith("gpt"):
    model = ChatOpenAI(api_key=OPENAI_API_KEY, model = MODEL)
    embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
else:
    model = OllamaLLM(model=MODEL)


In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

label_template = """
Your task is to analyze the provided document and intelligently split it into meaningful, self-contained chunks optimized for retrieval in a question-answering system.  

### **Agentic Chunking Guidelines**:
1️⃣ **Preserve Semantic Meaning** → Ensure each chunk is **logically complete**, avoiding mid-thought cutoffs.  
2️⃣ **Identify Key Propositions** → Extract **clear, well-formed** factual chunks aligned with **policies, procedures, or best practices**.  
3️⃣ **Maintain Cohesion** → Each chunk should be **self-explanatory**, requiring **minimal external context**.  
4️⃣ **Prioritize Answerability** → Structure chunks so they directly support **relevant user queries**.  
5️⃣ **Retain Critical Information** → Keep **tables, step-by-step instructions, or structured data** intact if they improve clarity.  
6️⃣ **Assign Metadata Tags** → Label each chunk with **relevant topics** for improved **search accuracy**.  

---
### **🛠 Additional Task: Generate Similar User Queries**
After chunking, analyze the key **topics, facts, and concepts** in the document and choose **one primary focus**. Then, generate **20 semantically similar questions** that users might ask to retrieve this information.  

### **💡 Question Generation Rules**:
✅ Use **natural language**  
✅ Occasionally include **typos or colloquialisms** to mimic real user behavior  
✅ Ensure **semantic similarity** without directly copying phrases from the document  
✅ All 20 questions must focus on the **same topic**  

---
### **✅ Expected Output Format**
```json
{{
  "chunks": [
    {{
      "chunk_number": 1,
      "chunk_content": "To reset an alarm device, hold the reset button for 10 seconds until the indicator light turns green. If the issue persists, contact support for troubleshooting.",
      "context_tags": ["Device Reset", "Troubleshooting"]
    }},
    {{
      "chunk_number": 2,
      "chunk_content": "Emergency calls are handled 24/7. If a call is received, the operator assesses the situation and dispatches emergency services if necessary.",
      "context_tags": ["Emergency Response", "Telecare"]
    }}
  ],
  "similar_questions": [
    "How do I reset my alarm if it's not working?",
    "My alarm won’t reset—what should I do?",
    "What’s the process for troubleshooting a device reset issue?",
    "Why is my emergency alarm still flashing red after a reset?",
    "Is there a way to restart the alarm without unplugging it?",
    "How long do I need to press the reset button on my alarm?",
    "Reset button not working—what now?",
    "Why is my emergency alarm not responding to resets?",
    "Steps to fix an alarm that won't reset?",
    "How do I fix my alarm if it’s stuck in reset mode?",
    "What’s the best way to reset a malfunctioning alarm?",
    "Do I need to unplug my alarm to reset it?",
    "Why does my alarm keep beeping after I try to reset it?",
    "Is there a special way to reset an emergency device?",
    "How to troubleshoot a reset issue on my device?",
    "My device won’t stop flashing—how do I fix it?",
    "What should I do if the reset instructions don’t work?",
    "How do I reset my emergency system manually?",
    "How can I fix a failed alarm reset?",
    "Why isn’t my reset button responding?"
  ]
}}

---
### Document
{document}
"""

label_prompt = ChatPromptTemplate.from_template(label_template)
# Uses the model selected above. Run with: chunking_chain.invoke({"document": pdf_text})
chunking_chain = label_prompt | model | JsonOutputParser()

In [ ]:
embeddings1 = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load_and_split()
pages

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load and extract text from PDF
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

# Combine text from all pages
full_text = " ".join([page.page_content for page in pages])

print(len(full_text.split()))  # Word count (approximate token count)


In [ ]:
import tiktoken

# Choose the tokenizer (e.g., for GPT-4, use "cl100k_base")
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(full_text)

print(len(tokens))  # Token count


Prompt Templates

In [ ]:
from langchain.prompts import PromptTemplate

template = """ 
You are helping Telecare Monitoring Representatives answerong the questions based on the context of the question. If you can't
answer the question, reply "I don't know. I only have a limited documents to check your answer against. Tell Titus to add more documents".

context: {context}

Question: {question}

"""

prompt = PromptTemplate.from_template(template)
print(prompt.format(context="Here is the context", question="Here is the question"))

In [ ]:
chain = prompt | model | parser
chain.invoke({
    "context": "Fall detection alarms use motion sensors and accelerometers to trigger an alert when a sudden fall is detected.",
    "question": "How does a fall detection alarm know someone has fallen?"
})

# Combining chains

We can combine different chains to create more complex workflows. For example, although this is not necessary in the present project we can implement it in translating it to Te Re Maori, lets create a second chain that translates the answer from the first chain into a different language.

Let's start by creating a new prompt template for the translation chain:

In [ ]:
## Use the following code if you want to translate the language to Te Reo Maori
from langchain_core.prompts import ChatPromptTemplate

translation_prompt = ChatPromptTemplate.from_template(
    "Translate {answer} to {language}")

What the above code does is, it creates a new chain after the previous one where a question and context ----> model ----> parser. The parser is now the output to translation_propmt (language) ----> model ----> parser ----> answer

* When we invoke a chain we keep in mind what we are invoking i.e., the question and a context which were then passed to prompt. There is trick anyways though which can be used as below

In [ ]:
chain.input_schema.model_json_schema()

* There are 3 options to deal with the warnings in the above code
1. You can update the code to chain.input_schema.model_json_schema()
2. Ignore warnings temporarily using 
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
3. check for uodates


DocArrayInMemorySearch search vector store which is in memory (local). In real world we wont be using in memory to store our vector embedding. we might be using pinecone or some other vector stores.

In [ ]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_documents(pages, embedding=embeddings1)

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document

retriever = BM25Retriever.from_documents(pages)

# Hybrid Search (Combining Dense + Sparse)
vector_retriever = vectorstore.as_retriever()

hybrid_retriever = EnsembleRetriever(retrievers=[vector_retriever, retriever])

hybrid_retriever.invoke("Medical Alarms, Mobile Alarms, TMRs, Alerts, Triage, Fall Detection, Panic Button, emergency Service Dispatch")

docs = hybrid_retriever.invoke("Medical Alarms, Mobile Alarms, TMRs, Alerts, Triage, Fall Detection, Panic Button, emergency Service Dispatch")
for doc in docs:
    print(doc.page_content)

retriever is component of langchain that will allow us to retrieve information from anywhere. We can create a retriever that pulls information from anywhere i.e., google etc but in this project we are retrieving information from the document which we declared in "pages" in the code above

In [ ]:
chain = prompt | model | parser

The prompt in the chain above is expecting a context and a question.
Context is going to come from the "retriever" but that retriever requires a question that we are invoking the chain with as below which is used in chain.invoke({"question": "What is Re-referencing?"}).
Runnable is the component that is going to generate a map that contains context and question. The first variable of the map (runnable) which is context which is going to come from a retriever. From the invoke we are going to grab the question, pipe that question into a retriever and the output of that is going to context.

In [ ]:
from operator import itemgetter

chain = ( 
{"context": itemgetter("question") | hybrid_retriever, 
 "question": itemgetter("question")}
| prompt
| model
| parser)

chain.invoke({"question": "What is a GO alarm?"})

In [ ]:
import gradio as gr

def gradio_interface(question):
    return chain.invoke({"question": question})


In [ ]:
# ✅ Improved UI
with gr.Blocks(css="""
    #submit-btn {
        font-size: 12px !important;
        padding: 5px 12px !important;
        min-width: 70px !important;
        height: 30px !important;
        border-radius: 5px !important;
    }
    #title {
        text-align: center;
        font-size: 24px;
        font-weight: bold;
        color: #006400;
    }
    #chat-area {
        background-color: #f8f9fa;
        padding: 10px;
        border-radius: 10px;
    }
""") as demo:
    gr.Markdown("<h1 id='title'>📟 Alarm Manual Q&A Chatbot</h1>")
    
    with gr.Row(elem_id="chat-area"):
        question = gr.Textbox(label="Ask a question", placeholder="Type your query here...", interactive=True)
        output = gr.Textbox(label="Answer", interactive=False)

    submit_btn = gr.Button("Send", elem_id="submit-btn")  # Small button
    submit_btn.click(gradio_interface, inputs=question, outputs=output)

In [ ]:
demo.launch()